# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 11_12_15_main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Geometry generation script
**Goal:** Generate the geometry dataset used as input for the Grasshopper structural analysis. The script accepts several variables and writes a CSV file containing the properties needed to reconstruct the digital geometry as CAD geometry.

**Inputs:**   
*   Base mesh definition
*   Allowed movement for different vertices
*   Divisions over the allowed movement

**Outputs:**

*   CSV file with sample ID, vertices, normalized coordinates, and edges
*   Centroid and PCA alignment are applied once in generation; do not repeat that normalization in Grasshopper

### Testing corners

In [1]:
import c11_params
import config
from c12_geometry_truss import get_corner_indices

# --- TEST ---
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 grid indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 grid indices: {grid_3x3}")

grid_4x4 = get_corner_indices(4, 4)
print(f"4x4 grid indices: {grid_4x4}")
# Expected: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> because there are 5 points per row

# In your loop:
corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] for example
print(corner_values)

parameters loaded from c:\Users\jaspe\Documents\PyRepo\thesis_generative_timber\c11_params.py
GRID: 2x2, EDGE_LENGTH: 3.0, LAYER_HEIGHT: 1.5, DIVISIONS: 8, NUM_SAMPLES: 20000

System loaded successfully.

Code is running locally from: thesis_generative_timber
Data connected to OneDrive: 2.2 - 2.4

GH data directory: C:\Users\jaspe\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\01_grasshopper_data
Raw data directory: C:\Users\jaspe\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\02_raw_data
Export directory: C:\Users\jaspe\OneDrive\06 Building Technology TU\2.2 - 2.4\60_Research_Exports

2x2 grid indices: {'bottom_left': 0, 'bottom_right': 2, 'top_left': 6, 'top_right': 8}
3x3 grid indices: {'bottom_left': 0, 'bottom_right': 3, 'top_left': 12, 'top_right': 15}
4x4 grid indices: {'bottom_left': 0, 'bottom_right': 4, 'top_left': 20, 'top_right': 24}
dict_values([0, 2, 6, 8])


## 1.2 Executing and saving dataset

In [2]:
import pandas as pd
import numpy as np
import config
import c11_params
from c12_geometry_truss import generate_vertices, generate_edges

# --- RUN ---
final_vertices = generate_vertices(c11_params.NUM_SAMPLES)
final_edges = generate_edges(c11_params.NUM_SAMPLES, c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

# --- SAVE ---
final_vertices_name = f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv"
final_vertices.to_csv(config.RAW_DATA_PATH / final_vertices_name, index=False)
final_edges.to_csv(config.RAW_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_edges.csv", index=False)

print(f"Generation complete for {c11_params.NUM_SAMPLES} samples.")
print(f"Grid size: {c11_params.GRID}")

print("Vertices shape:", final_vertices.shape)
print("Edges shape:", final_edges.shape)

print(f"\nTotal rows in vertices CSV: {len(final_vertices)}")
print("\nExample output (first 5 rows of sample 0):")
print(final_vertices.head(5))
print("\nExample output (first 5 rows of sample 1):")
print(final_vertices[final_vertices['sample_id'] == 1].head(5))

print(f"\nTotal rows in edges CSV: {len(final_edges)}")
print("\nExample output (first 5 rows of sample 0):")
print(final_edges.head(5))
print("\nExample output (first 5 rows of sample 1):")
print(final_edges[final_edges['sample_id'] == 1].head(5))

Generation complete for 20000 samples.
Grid size: 2x2
Vertices shape: (260000, 7)
Edges shape: (640000, 4)

Total rows in vertices CSV: 260000

Example output (first 5 rows of sample 0):
   sample_id vertex_index layer attribute      x      y     z
0          0           v0   top   support -3.009 -2.968  0.49
1          0           v1   top      load -1.134 -2.968  0.49
2          0           v2   top   support  2.991 -2.968  0.49
3          0           v3   top      load -3.009 -0.343  0.49
4          0           v4   top      load -0.759  0.782  0.49

Example output (first 5 rows of sample 1):
    sample_id vertex_index layer attribute      x      y      z
13          1           v0   top   support -3.064 -3.144  0.375
14          1           v1   top      load  0.311 -3.144  0.375
15          1           v2   top   support  2.936 -3.144  0.375
16          1           v3   top      load -3.064  0.606  0.375
17          1           v4   top      load -0.064  0.606  0.375

Total rows i

## 1.5 DESIGN DOMAIN
two output from the geometry script need to be transferred to json files for further use in the script, these are:
*   Search Space, this is the space where the vertices can move, this is nessecary for the optimizing of the geometry so the optimizer knows where it can and cant go.
*   Edge Index, the edge index is nessecary for the training of the surrogate model, this is used because the relationship of the vertices is the same vor every structure, also lowers this the number of features (x) used in training for a better generalisation.

In [3]:
import json
import config
import c11_params
from c12_geometry_truss import generate_edges, define_search_space

# --- RUN AND REVIEW ---
# Use the variables from your earlier configuration
search_space = define_search_space(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y, c11_params.DIVISIONS, c11_params.EDGE_LENGTH)
df_topology = generate_edges(num_samples=1, cells_x=c11_params.GRID_CELLS_X, cells_y=c11_params.GRID_CELLS_Y)
edge_index = df_topology[['V1', 'V2']].values.T.tolist()

print(f"The search space contains {len(search_space)} independent variables (the controls the AI can adjust).")
print(f"Topology generated. Number of unique connections (edges): {len(df_topology)}")
print("\nExample of how the computer sees this:")

# Show the first 5 variables in a readable format
for var_name, bounds in list(search_space.items())[:5]:
    print(f"- {var_name}: {bounds}")

print("\nGenerated edge_index (first 5 connections):")
print(f"Source nodes (V1): {edge_index[0][:5]}")
print(f"Target nodes (V2): {edge_index[1][:5]}")

# Save the dictionaries as JSON files
json_search_space_path = config.DATA_IO_PATH / "search_space.json"
json_edge_index_path = config.DATA_IO_PATH / "edge_index.json"
with open(json_search_space_path, 'w') as f:
    json.dump(search_space, f, indent=4) # indent=4 makes it easy to read
with open(json_edge_index_path, 'w') as f:
    json.dump(edge_index, f)

print(f"Search space successfully saved as '{json_search_space_path}'")
print(f"\nEdge index (as a plain list) successfully saved to '{json_edge_index_path}'.")

The search space contains 18 independent variables (the controls the AI can adjust).
Topology generated. Number of unique connections (edges): 32

Example of how the computer sees this:
- v1_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v3_shift_y: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v4_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v4_shift_y: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v5_shift_y: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}

Generated edge_index (first 5 connections):
Source nodes (V1): [0, 0, 1, 1, 2]
Target nodes (V2): [1, 3, 2, 4, 5]
Search space successfully saved as 'C:\Users\jaspe\Documents\PyRepo\thesis_generative_timber\02_data_io\search_space.json'

Edge index (as a plain list) successfully saved to 'C:\Users\jaspe\Documents\PyRepo\thesis_gene

## Average Length
Here the average length of the geometry is calculated for the generation of the timber stock

In [4]:
import config
import json
from c23_reconstruction import calculate_representative_beam_length_from_files

result = calculate_representative_beam_length_from_files(
    vertices_csv_path= config.RAW_DATA_PATH / final_vertices_name,
    edge_index_json_path= config.DATA_IO_PATH / "edge_index.json",
    sample_count=5,
    random_state=42
)

print(result["representative_length_m"], result["representative_length_mm"])
print(result["selected_sample_ids"])

with open(config.DATA_IO_PATH / "representative_beam_length.json", 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

2.8905104844016405 2890.5104844016405
[3648, 819, 9012, 8024, 7314]
